In [1]:
"""
=============================================================================
TUGAS FISIKA GALAKSI: Lengan Spiral Galaksi – Distribusi OB dan OC
=============================================================================
Data:
  • Gugus Terbuka (OC) : ncovocc_final.csv      (NCOVOCC, Dias et al. 2003)
  • Bintang OB         : bintang_ob_final.csv   (Melnik & Efremov 1995)

Cara menjalankan: python3 tugas_lengkap.py
Output: 6 file gambar (.png) + 2 file CSV bersih
=============================================================================
"""

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')          # Hapus baris ini jika ingin tampil interaktif
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams.update({
    'font.size'       : 11,
    'axes.titlesize'  : 12,
    'axes.labelsize'  : 11,
    'legend.fontsize' : 9,
})

X0 = 8.0   # Jarak Matahari ke Pusat Galaksi (kpc)


In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# BAGIAN A: PEMBERSIHAN DATA
# ─────────────────────────────────────────────────────────────────────────────

print("=" * 65)
print("BAGIAN A: PEMBERSIHAN DATA")
print("=" * 65)

# ── A1. Muat data mentah ────────────────────────────────────────────────────
oc_raw = pd.read_csv('ncovocc_final.csv')
ob_raw = pd.read_csv('bintang_ob_final.csv')

print(f"\n[OC] Baris awal : {len(oc_raw)}  |  Kolom: {len(oc_raw.columns)}")
print(f"[OB] Baris awal : {len(ob_raw)}  |  Kolom: {len(ob_raw.columns)}")

# ── A2. Bersihkan dataset OC (Gugus Terbuka) ───────────────────────────────
print("\n--- Membersihkan OC ---")
oc = oc_raw.copy()

# C1: Strip whitespace pada kolom string
for kol in ['Cluster', 'Class', 'TrType', 'K14', 'WEBDA', 'Lynga']:
    if kol in oc.columns:
        oc[kol] = oc[kol].astype(str).str.strip()
oc.replace('nan', np.nan, inplace=True)

# C2: Hapus duplikat (FSR 1496 muncul 2×)
n_dup_oc = oc.duplicated().sum()
oc = oc.drop_duplicates(keep='first').reset_index(drop=True)
print(f"  Duplikat dihapus   : {n_dup_oc} baris")

# C3: Tandai baris tanpa Age
oc['has_age'] = oc['Age'].notna()
print(f"  NaN pada Age       : {(~oc['has_age']).sum()} baris (ditandai has_age=False)")

# C4: Tandai outlier jarak
q1, q3  = oc['Dist'].quantile(0.25), oc['Dist'].quantile(0.75)
batas   = q3 + 3 * (q3 - q1)
oc['dist_outlier'] = oc['Dist'] > batas
print(f"  Outlier Dist >     : {batas:.0f} pc → {oc['dist_outlier'].sum()} baris (ditandai)")
print(f"  Baris OC bersih    : {len(oc)}")

# ── A3. Bersihkan dataset OB (Bintang OB) ──────────────────────────────────
print("\n--- Membersihkan OB ---")
ob = ob_raw.copy()

# C1: Strip whitespace kolom string
for kol in ['Name', 'SpType', 'AssBH', 'AssME']:
    if kol in ob.columns:
        ob[kol] = ob[kol].astype(str).str.strip()
ob.replace('nan', np.nan, inplace=True)

# C2: Hapus duplikat
n_dup_ob = ob.duplicated().sum()
ob = ob.drop_duplicates(keep='first').reset_index(drop=True)
print(f"  Duplikat dihapus   : {n_dup_ob} baris")

# C3: Validasi rBH (jarak harus > 0)
invalid_rbh = (ob['rBH'] <= 0).sum()
if invalid_rbh > 0:
    ob = ob[ob['rBH'] > 0].reset_index(drop=True)
    print(f"  rBH ≤ 0 dihapus    : {invalid_rbh} baris")
else:
    print(f"  Validasi rBH       : ✓ semua nilai positif")
print(f"  Baris OB bersih    : {len(ob)}")

# ── A4. Simpan CSV bersih ───────────────────────────────────────────────────
oc.to_csv('oc_bersih.csv', index=False)
ob.to_csv('ob_bersih.csv', index=False)
print(f"\n  Disimpan: oc_bersih.csv ({len(oc)} baris)")
print(f"  Disimpan: ob_bersih.csv ({len(ob)} baris)")


BAGIAN A: PEMBERSIHAN DATA

[OC] Baris awal : 1804  |  Kolom: 27
[OB] Baris awal : 1390  |  Kolom: 13

--- Membersihkan OC ---
  Duplikat dihapus   : 1 baris
  NaN pada Age       : 25 baris (ditandai has_age=False)
  Outlier Dist >     : 7100 pc → 17 baris (ditandai)
  Baris OC bersih    : 1803

--- Membersihkan OB ---
  Duplikat dihapus   : 8 baris
  Validasi rBH       : ✓ semua nilai positif
  Baris OB bersih    : 1382

  Disimpan: oc_bersih.csv (1803 baris)
  Disimpan: ob_bersih.csv (1382 baris)


In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# LANGKAH 1: HITUNG KOORDINAT GALAKTOSENTRIS (X, Y, Z)
# ─────────────────────────────────────────────────────────────────────────────
# Persamaan:
#   X = X₀ − d·cos(b)·cos(l)      X₀ = 8.0 kpc
#   Y = d·cos(b)·sin(l)
#   Z = d·sin(b)
#
# Untuk OC: d dalam kpc = Dist(pc) / 1000
# Untuk OB: d dalam kpc = rBH (sudah dalam kpc)

print("\n" + "=" * 65)
print("LANGKAH 1: Koordinat Galaktosentris X, Y, Z")
print("=" * 65)

# ── OC ──
l_oc = np.radians(oc['_Glon'])
b_oc = np.radians(oc['_Glat'])
d_oc = oc['Dist'] / 1000.0          # parsec → kpc

oc['X'] = X0 - d_oc * np.cos(b_oc) * np.cos(l_oc)
oc['Y'] = d_oc * np.cos(b_oc) * np.sin(l_oc)
oc['Z'] = d_oc * np.sin(b_oc)

# ── OB ──
l_ob = np.radians(ob['GLON'])
b_ob = np.radians(ob['GLAT'])
d_ob = ob['rBH']                    # sudah dalam kpc

ob['X'] = X0 - d_ob * np.cos(b_ob) * np.cos(l_ob)
ob['Y'] = d_ob * np.cos(b_ob) * np.sin(l_ob)
ob['Z'] = d_ob * np.sin(b_ob)

print(f"\n[OC] X: {oc['X'].min():.2f} – {oc['X'].max():.2f} kpc")
print(f"[OC] Y: {oc['Y'].min():.2f} – {oc['Y'].max():.2f} kpc")
print(f"[OC] Z: {oc['Z'].min():.3f} – {oc['Z'].max():.3f} kpc")
print(f"\n[OB] X: {ob['X'].min():.2f} – {ob['X'].max():.2f} kpc")
print(f"[OB] Y: {ob['Y'].min():.2f} – {ob['Y'].max():.2f} kpc")
print(f"[OB] Z: {ob['Z'].min():.3f} – {ob['Z'].max():.3f} kpc")



LANGKAH 1: Koordinat Galaktosentris X, Y, Z

[OC] X: 0.43 – 16.47 kpc
[OC] Y: -8.40 – 14.46 kpc
[OC] Z: -0.300 – 0.300 kpc

[OB] X: 3.68 – 11.00 kpc
[OB] Y: -5.92 – 4.74 kpc
[OB] Z: -0.189 – 0.167 kpc


In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# LANGKAH 2: PLOT Z vs BUJUR GALAKTIK l  (OC dan OB dibedakan)
# ─────────────────────────────────────────────────────────────────────────────
# Tujuan: Apakah kedua populasi benar-benar merupakan populasi piringan Galaksi?
# Populasi piringan → Z kecil, tersebar tipis di sekitar Z = 0.
#
# Cara membedakan OC vs OB di plot:
#   • OC: lingkaran biru kecil (○), semi-transparan
#   • OB: tanda silang merah (×), sedikit lebih besar

print("\n" + "=" * 65)
print("LANGKAH 2: Plot Z vs Bujur Galaktik l (OC + OB)")
print("=" * 65)

fig2, ax2 = plt.subplots(figsize=(13, 5))

# Plot OC (lingkaran biru)
ax2.scatter(
    oc['_Glon'], oc['Z'],
    marker='o',           # simbol lingkaran untuk OC
    s=5,                  # ukuran titik kecil
    c='steelblue',
    alpha=0.5,
    label=f'Gugus Terbuka / OC (n={len(oc)})',
    zorder=2
)

# Plot OB (silang merah)
ax2.scatter(
    ob['GLON'], ob['Z'],
    marker='x',           # simbol silang untuk OB
    s=15,
    c='crimson',
    alpha=0.6,
    linewidths=0.7,
    label=f'Bintang OB (n={len(ob)})',
    zorder=3
)

# Garis referensi
ax2.axhline(0,     color='black', lw=1.0, ls='--', alpha=0.5,
            label='Bidang galaksi (Z = 0)')
ax2.axhline( 0.3,  color='gray',  lw=1.0, ls=':',  alpha=0.8,
            label='Batas seleksi |Z| = 0.3 kpc')
ax2.axhline(-0.3,  color='gray',  lw=1.0, ls=':',  alpha=0.8)

ax2.set_xlabel('Bujur Galaktik $l$ (derajat)')
ax2.set_ylabel('Z (kpc)')
ax2.set_title('Langkah 2: Ketinggian Z terhadap Bujur Galaktik\n'
              'Lingkaran biru = Gugus Terbuka (OC)  |  Silang merah = Bintang OB')
ax2.set_xlim(0, 360)
ax2.set_ylim(-0.6, 0.6)
ax2.legend(loc='upper right')
ax2.grid(True, alpha=0.25)

fig2.tight_layout()
fig2.savefig('langkah2_Z_vs_Glon.png', dpi=150, bbox_inches='tight')
plt.close(fig2)

# Analisis
persen_oc = (oc['Z'].abs() < 0.3).mean() * 100
persen_ob = (ob['Z'].abs() < 0.3).mean() * 100
print(f"\n[OC] Fraksi |Z| < 0.3 kpc : {persen_oc:.1f}%")
print(f"[OB] Fraksi |Z| < 0.3 kpc : {persen_ob:.1f}%")
print("→ Keduanya terkonsentrasi di dekat bidang galaksi → populasi piringan tipis.")
print("Disimpan: langkah2_Z_vs_Glon.png")



LANGKAH 2: Plot Z vs Bujur Galaktik l (OC + OB)

[OC] Fraksi |Z| < 0.3 kpc : 100.0%
[OB] Fraksi |Z| < 0.3 kpc : 100.0%
→ Keduanya terkonsentrasi di dekat bidang galaksi → populasi piringan tipis.
Disimpan: langkah2_Z_vs_Glon.png


In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# LANGKAH 3: SELEKSI OBJEK DENGAN |Z| < 0.3 kpc
# ─────────────────────────────────────────────────────────────────────────────
# Fokuskan analisis hanya pada piringan tipis Galaksi.

print("\n" + "=" * 65)
print("LANGKAH 3: Seleksi |Z| < 0.3 kpc")
print("=" * 65)

oc_disk = oc[oc['Z'].abs() < 0.3].copy()
ob_disk = ob[ob['Z'].abs() < 0.3].copy()

print(f"[OC] Sebelum: {len(oc)}  → Setelah: {len(oc_disk)}")
print(f"[OB] Sebelum: {len(ob)}  → Setelah: {len(ob_disk)}")

# Sub-dataframe OC berdasarkan usia (untuk Langkah 5)
has_age    = oc_disk['has_age']
oc_muda    = oc_disk[has_age & (oc_disk['Age'] <  7.5)]
oc_sedang  = oc_disk[has_age & (oc_disk['Age'] >= 7.5) & (oc_disk['Age'] < 8.5)]
oc_tua     = oc_disk[has_age & (oc_disk['Age'] >= 8.5)]

print(f"\n[OC] Muda   log t < 7.5      : {len(oc_muda)} gugus  (< 32 Myr)")
print(f"[OC] Sedang 7.5 ≤ log t < 8.5: {len(oc_sedang)} gugus")
print(f"[OC] Tua    log t ≥ 8.5      : {len(oc_tua)} gugus  (> 316 Myr)")



LANGKAH 3: Seleksi |Z| < 0.3 kpc
[OC] Sebelum: 1803  → Setelah: 1803
[OB] Sebelum: 1382  → Setelah: 1382

[OC] Muda   log t < 7.5      : 364 gugus  (< 32 Myr)
[OC] Sedang 7.5 ≤ log t < 8.5: 593 gugus
[OC] Tua    log t ≥ 8.5      : 821 gugus  (> 316 Myr)


In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# LANGKAH 4: PLOT POSISI BIDANG GALAKSI – SEMUA OC  (+ OB sebagai latar)
# ─────────────────────────────────────────────────────────────────────────────
# Plot X vs Y, dengan batas: 0 < X < 12 kpc, -6 < Y < 6 kpc.
# OC dan OB dibedakan warna dan simbol.

print("\n" + "=" * 65)
print("LANGKAH 4: Plot Posisi Bidang Galaksi – Semua OC + OB")
print("=" * 65)

def filter_xy(df):
    """Filter objek dalam batas plot soal."""
    m = (df['X'] > 0) & (df['X'] < 12) & (df['Y'] > -6) & (df['Y'] < 6)
    return df[m]

oc_plot4 = filter_xy(oc_disk)
ob_plot4 = filter_xy(ob_disk)
print(f"[OC] Dalam batas plot: {len(oc_plot4)}")
print(f"[OB] Dalam batas plot: {len(ob_plot4)}")

fig4, ax4 = plt.subplots(figsize=(9, 9))

# OB di bawah (latar)
ax4.scatter(ob_plot4['Y'], ob_plot4['X'],
            marker='x', s=18, c='crimson', alpha=0.5,
            linewidths=0.8, zorder=2,
            label=f'Bintang OB (n={len(ob_plot4)})')

# OC di atas
ax4.scatter(oc_plot4['Y'], oc_plot4['X'],
            marker='o', s=8, c='steelblue', alpha=0.5,
            zorder=3,
            label=f'Gugus Terbuka / OC (n={len(oc_plot4)})')

# Matahari dan GC
ax4.plot(0, X0, '*', ms=18, color='gold', mec='darkorange',
         mew=1.5, zorder=6, label='☀ Matahari')
ax4.plot(0,  0, 'o', ms=10, color='black', zorder=6,
         label='● Pusat Galaksi (GC)')

# Lingkaran galaktosentris
for r in [2, 4, 6, 8, 10]:
    circ = plt.Circle((0, 0), r, color='gray',
                      fill=False, lw=0.6, ls='--', alpha=0.3)
    ax4.add_patch(circ)
    ax4.text(r * 0.68, r * 0.68 + 0.15, f'{r} kpc',
             fontsize=7.5, color='gray', alpha=0.6)

ax4.set_xlabel('Y (kpc) → arah $l = 90°$')
ax4.set_ylabel('X (kpc) → arah $l = 0°$ (GC)')
ax4.set_title('Langkah 4: Distribusi OC dan OB pada Bidang Galaksi\n'
              '(semua usia, |Z| < 0.3 kpc, tanpa model spiral)')
ax4.set_xlim(-6, 6)
ax4.set_ylim(0, 12)
ax4.set_aspect('equal')
ax4.legend(loc='upper right')
ax4.grid(True, alpha=0.2)

fig4.tight_layout()
fig4.savefig('langkah4_XY_semua.png', dpi=150, bbox_inches='tight')
plt.close(fig4)
print("Disimpan: langkah4_XY_semua.png")



LANGKAH 4: Plot Posisi Bidang Galaksi – Semua OC + OB
[OC] Dalam batas plot: 1753
[OB] Dalam batas plot: 1382
Disimpan: langkah4_XY_semua.png


In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# LANGKAH 5: PLOT PER KELOMPOK USIA OC  (+ OB sebagai perbandingan)
# ─────────────────────────────────────────────────────────────────────────────
# Dua panel:
#   Kiri  : OC muda (log t < 7.5)  + OB
#   Kanan : OC sedang (7.5 ≤ log t < 8.5) + OB

print("\n" + "=" * 65)
print("LANGKAH 5: Plot Per Kelompok Usia OC  +  OB")
print("=" * 65)

oc_muda_p   = filter_xy(oc_muda)
oc_sedang_p = filter_xy(oc_sedang)
ob_p5       = filter_xy(ob_disk)

fig5, axes5 = plt.subplots(1, 2, figsize=(16, 8), sharey=True)
fig5.suptitle(
    'Langkah 5: Distribusi OC Berdasarkan Usia + Bintang OB\n'
    'Kiri: OC muda (log t < 7.5)  |  Kanan: OC sedang (7.5 ≤ log t < 8.5)',
    fontsize=12, fontweight='bold')

for ax, oc_grp, judul in [
    (axes5[0], oc_muda_p,
     f'OC Muda  log t < 7.5\n(usia < 32 Myr, n={len(oc_muda_p)})'),
    (axes5[1], oc_sedang_p,
     f'OC Sedang  7.5 ≤ log t < 8.5\n(32–316 Myr, n={len(oc_sedang_p)})'),
]:
    # OB (latar)
    ax.scatter(ob_p5['Y'], ob_p5['X'],
               marker='x', s=12, c='crimson', alpha=0.4,
               linewidths=0.7, zorder=2,
               label=f'Bintang OB (n={len(ob_p5)})')
    # OC kelompok usia
    ax.scatter(oc_grp['Y'], oc_grp['X'],
               marker='o', s=20, c='steelblue', alpha=0.8,
               zorder=3, label=judul.split('\n')[0])

    ax.plot(0, X0, '*', ms=14, color='gold', mec='darkorange',
            mew=1.5, zorder=6, label='☀ Matahari')
    ax.plot(0,  0, 'o', ms=8, color='black', zorder=6, label='● GC')

    for r in [2, 4, 6, 8, 10]:
        circ = plt.Circle((0, 0), r, color='gray',
                          fill=False, lw=0.5, ls='--', alpha=0.3)
        ax.add_patch(circ)

    ax.set_xlabel('Y (kpc)')
    ax.set_ylabel('X (kpc)')
    ax.set_title(judul, fontsize=10)
    ax.set_xlim(-6, 6)
    ax.set_ylim(0, 12)
    ax.set_aspect('equal')
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(True, alpha=0.2)

fig5.tight_layout()
fig5.savefig('langkah5_XY_usia.png', dpi=150, bbox_inches='tight')
plt.close(fig5)
print(f"OC muda dalam plot   : {len(oc_muda_p)}")
print(f"OC sedang dalam plot : {len(oc_sedang_p)}")
print(f"OB dalam plot        : {len(ob_p5)}")
print("Disimpan: langkah5_XY_usia.png")



LANGKAH 5: Plot Per Kelompok Usia OC  +  OB
OC muda dalam plot   : 356
OC sedang dalam plot : 584
OB dalam plot        : 1382
Disimpan: langkah5_XY_usia.png


In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# LANGKAH 6: OVERPLOT MODEL LENGAN SPIRAL  r(θ) = r_i · exp(k·(θ − θ₀))
# ─────────────────────────────────────────────────────────────────────────────
# Model logaritmik dari Dias & Lépine (2005).
#
# Parameter:
#   r_i  : radius awal lengan dari GC (kpc)
#   k    : tan(sudut pitch p);  Bimasakti p ≈ 10°–15° → k ≈ 0.18–0.27
#   θ₀   : offset fase sudut awal (radian)
#   θ    : sudut polar galaktosentris, diukur dari arah GC→Matahari
#           θ=0 → arah +X (Sun), θ=π/2 → arah +Y
#
# Konversi ke koordinat plot:
#   x_plot = r·cos(θ)
#   y_plot = r·sin(θ)

print("\n" + "=" * 65)
print("LANGKAH 6: Overplot Model Lengan Spiral (Dias & Lépine 2005)")
print("=" * 65)

def spiral(r_i, k, theta0, theta_arr):
    """
    Hitung kurva lengan spiral logaritmik.
    Mengembalikan (x_kpc, y_kpc) dalam koordinat galaktosentris.
    """
    r    = r_i * np.exp(k * (theta_arr - theta0))
    x_sp = r * np.cos(theta_arr)
    y_sp = r * np.sin(theta_arr)
    return x_sp, y_sp

theta = np.linspace(-np.pi, 2.5 * np.pi, 3000)

# Parameter 5 lengan spiral utama Bimasakti
# (Nama, r_i kpc, k=tan(pitch), θ₀ rad, warna)
LENGAN = [
    ('Norma–Cygnus',  3.48, 0.26, 0.00, 'purple'),
    ('Scutum–Crux',   4.90, 0.26, 0.00, 'darkorange'),
    ('Carina–Sag.',   6.04, 0.23, 0.00, 'dodgerblue'),
    ('Local Arm',     7.70, 0.18, 4.80, 'forestgreen'),
    ('Perseus',       9.90, 0.21, 4.71, 'red'),
]

print("\nParameter lengan spiral:")
print(f"{'Nama':<18} {'r_i (kpc)':>10} {'k':>8} {'θ₀ (rad)':>10} {'Warna':<12}")
print("-" * 62)
for nama, r_i, k, t0, col in LENGAN:
    print(f"{nama:<18} {r_i:>10.2f} {k:>8.3f} {t0:>10.2f} {col:<12}")

def tambahkan_spiral(ax):
    """Tambahkan overplot model lengan spiral ke axes yang diberikan."""
    for nama, r_i, k, t0, col in LENGAN:
        x_sp, y_sp = spiral(r_i, k, t0, theta)
        mask = (x_sp > 0) & (x_sp < 12) & (y_sp > -6) & (y_sp < 6)
        ax.plot(y_sp[mask], x_sp[mask],
                color=col, lw=2.0, alpha=0.85,
                label=nama, zorder=5)

def tambahkan_matahari_gc(ax):
    """Tambahkan tanda Matahari dan Pusat Galaksi."""
    ax.plot(0, X0, '*', ms=18, color='gold', mec='darkorange',
            mew=1.5, zorder=8, label='☀ Matahari')
    ax.plot(0,  0, 'o', ms=10, color='black', zorder=8,
            label='● Pusat Galaksi')

def tambahkan_lingkaran(ax):
    """Tambahkan lingkaran galaktosentris sebagai panduan jarak."""
    for r in [2, 4, 6, 8, 10]:
        circ = plt.Circle((0, 0), r, color='gray',
                          fill=False, lw=0.6, ls='--', alpha=0.3)
        ax.add_patch(circ)
        ax.text(r * 0.68, r * 0.68 + 0.15,
                f'{r} kpc', fontsize=7.5, color='gray', alpha=0.6)

# ── Plot 6A: OC Muda + OB + Spiral ──────────────────────────────────────────
fig6a, ax6a = plt.subplots(figsize=(10, 10))

ax6a.scatter(filter_xy(oc_tua)['Y'],   filter_xy(oc_tua)['X'],
             marker='o', s=5, c='lightgray', alpha=0.35, zorder=1,
             label=f'OC Tua log t≥8.5 (latar, n={len(filter_xy(oc_tua))})')

ax6a.scatter(ob_p5['Y'], ob_p5['X'],
             marker='x', s=18, c='crimson', alpha=0.55,
             linewidths=0.9, zorder=3,
             label=f'Bintang OB (n={len(ob_p5)})')

ax6a.scatter(oc_muda_p['Y'], oc_muda_p['X'],
             marker='o', s=22, c='steelblue', alpha=0.85, zorder=4,
             label=f'OC Muda log t<7.5 (n={len(oc_muda_p)})')

tambahkan_spiral(ax6a)
tambahkan_matahari_gc(ax6a)
tambahkan_lingkaran(ax6a)

ax6a.set_xlabel('Y (kpc) → arah $l = 90°$')
ax6a.set_ylabel('X (kpc) → arah $l = 0°$ (GC)')
ax6a.set_title('Langkah 6a: OC Muda (log t < 7.5) + Bintang OB\n'
               '+ Model Lengan Spiral Logaritmik (Dias & Lépine 2005)',)
ax6a.set_xlim(-6, 6); ax6a.set_ylim(0, 12)
ax6a.set_aspect('equal')
ax6a.legend(loc='upper right', fontsize=8, framealpha=0.88)
ax6a.grid(True, alpha=0.2)
fig6a.tight_layout()
fig6a.savefig('langkah6a_muda_ob_spiral.png', dpi=150, bbox_inches='tight')
plt.close(fig6a)
print("\nDisimpan: langkah6a_muda_ob_spiral.png")

# ── Plot 6B: Semua kelompok OC + OB + Spiral ─────────────────────────────────
fig6b, ax6b = plt.subplots(figsize=(10, 10))

ax6b.scatter(filter_xy(oc_tua)['Y'],    filter_xy(oc_tua)['X'],
             marker='o', s=5, c='lightgray', alpha=0.3, zorder=1,
             label=f'OC Tua ≥8.5 (n={len(filter_xy(oc_tua))})')
ax6b.scatter(oc_sedang_p['Y'],          oc_sedang_p['X'],
             marker='o', s=10, c='cornflowerblue', alpha=0.55, zorder=2,
             label=f'OC Sedang 7.5–8.5 (n={len(oc_sedang_p)})')
ax6b.scatter(oc_muda_p['Y'],            oc_muda_p['X'],
             marker='o', s=20, c='steelblue', alpha=0.85, zorder=3,
             label=f'OC Muda <7.5 (n={len(oc_muda_p)})')
ax6b.scatter(ob_p5['Y'],                ob_p5['X'],
             marker='x', s=18, c='crimson', alpha=0.5,
             linewidths=0.9, zorder=4,
             label=f'Bintang OB (n={len(ob_p5)})')

tambahkan_spiral(ax6b)
tambahkan_matahari_gc(ax6b)
tambahkan_lingkaran(ax6b)

ax6b.set_xlabel('Y (kpc) → arah $l = 90°$')
ax6b.set_ylabel('X (kpc) → arah $l = 0°$ (GC)')
ax6b.set_title('Langkah 6b: Semua Kelompok OC + Bintang OB\n'
               '+ Model Lengan Spiral Logaritmik (Dias & Lépine 2005)')
ax6b.set_xlim(-6, 6); ax6b.set_ylim(0, 12)
ax6b.set_aspect('equal')
ax6b.legend(loc='upper right', fontsize=8, framealpha=0.88)
ax6b.grid(True, alpha=0.2)
fig6b.tight_layout()
fig6b.savefig('langkah6b_semua_ob_spiral.png', dpi=150, bbox_inches='tight')
plt.close(fig6b)
print("Disimpan: langkah6b_semua_ob_spiral.png")



LANGKAH 6: Overplot Model Lengan Spiral (Dias & Lépine 2005)

Parameter lengan spiral:
Nama                r_i (kpc)        k   θ₀ (rad) Warna       
--------------------------------------------------------------
Norma–Cygnus             3.48    0.260       0.00 purple      
Scutum–Crux              4.90    0.260       0.00 darkorange  
Carina–Sag.              6.04    0.230       0.00 dodgerblue  
Local Arm                7.70    0.180       4.80 forestgreen 
Perseus                  9.90    0.210       4.71 red         

Disimpan: langkah6a_muda_ob_spiral.png
Disimpan: langkah6b_semua_ob_spiral.png


In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# RINGKASAN AKHIR
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("RINGKASAN AKHIR")
print("=" * 65)
print(f"""
Dataset setelah pembersihan:
  OC: {len(oc)} gugus  |  OB: {len(ob)} bintang

Langkah 2 – Z vs Bujur Galaktik:
  OC: {persen_oc:.1f}% berada dalam |Z| < 0.3 kpc → populasi piringan tipis
  OB: {persen_ob:.1f}% berada dalam |Z| < 0.3 kpc → populasi piringan tipis

Langkah 3 – Seleksi |Z| < 0.3 kpc:
  OC disk: {len(oc_disk)} gugus  |  OB disk: {len(ob_disk)} bintang

Langkah 5 – Kelompok Usia OC:
  Muda  (log t < 7.5)     : {len(oc_muda)} gugus  ← penanda spiral terbaik
  Sedang (7.5–8.5)        : {len(oc_sedang)} gugus
  Tua   (log t ≥ 8.5)     : {len(oc_tua)} gugus  ← tersebar, tidak jelas spiral

Langkah 6 – Model Spiral:
  r(θ) = r_i · exp(k · (θ − θ₀))
  5 lengan dimodelkan: Norma, Scutum-Crux, Carina-Sag, Local Arm, Perseus
  OC muda + OB sama-sama mengelompok di sekitar lengan Carina-Sag & Perseus.

Output file:
  oc_bersih.csv               → Data OC bersih
  ob_bersih.csv               → Data OB bersih
  langkah2_Z_vs_Glon.png      → Z vs l (OC + OB)
  langkah4_XY_semua.png       → X vs Y semua gugus + OB
  langkah5_XY_usia.png        → X vs Y per usia + OB
  langkah6a_muda_ob_spiral.png→ OC muda + OB + spiral
  langkah6b_semua_ob_spiral.png→ Semua OC + OB + spiral
""")



RINGKASAN AKHIR

Dataset setelah pembersihan:
  OC: 1803 gugus  |  OB: 1382 bintang

Langkah 2 – Z vs Bujur Galaktik:
  OC: 100.0% berada dalam |Z| < 0.3 kpc → populasi piringan tipis
  OB: 100.0% berada dalam |Z| < 0.3 kpc → populasi piringan tipis

Langkah 3 – Seleksi |Z| < 0.3 kpc:
  OC disk: 1803 gugus  |  OB disk: 1382 bintang

Langkah 5 – Kelompok Usia OC:
  Muda  (log t < 7.5)     : 364 gugus  ← penanda spiral terbaik
  Sedang (7.5–8.5)        : 593 gugus
  Tua   (log t ≥ 8.5)     : 821 gugus  ← tersebar, tidak jelas spiral

Langkah 6 – Model Spiral:
  r(θ) = r_i · exp(k · (θ − θ₀))
  5 lengan dimodelkan: Norma, Scutum-Crux, Carina-Sag, Local Arm, Perseus
  OC muda + OB sama-sama mengelompok di sekitar lengan Carina-Sag & Perseus.

Output file:
  oc_bersih.csv               → Data OC bersih
  ob_bersih.csv               → Data OB bersih
  langkah2_Z_vs_Glon.png      → Z vs l (OC + OB)
  langkah4_XY_semua.png       → X vs Y semua gugus + OB
  langkah5_XY_usia.png        → X vs Y

In [10]:
# ─────────────────────────────────────────────────────────────────────────────
# TAMBAHAN UNTUK NOMOR 7 DAN 8: ANALISIS TRACER & KUADRAN
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("ANALISIS TAMBAHAN (NOMOR 7 & 8)")
print("=" * 65)

# --- Analisis Nomor 7: Sebaran Z ---
# Membandingkan seberapa "rapat" mereka di bidang piringan (Z = 0)
std_z_oc = oc_muda['Z'].std()
std_z_ob = ob_disk['Z'].std()

print("--- Data Pendukung No. 7: Karakteristik Tracer ---")
print(f"Sebaran (Std Dev) sumbu Z OC Muda : {std_z_oc:.3f} kpc")
print(f"Sebaran (Std Dev) sumbu Z Btg OB  : {std_z_ob:.3f} kpc")
print("Semakin kecil nilainya, semakin objek tersebut menempel ketat pada piringan.")

# --- Analisis Nomor 8: Kelengkapan Kuadran ---
# Membagi distribusi tracer berdasarkan Bujur Galaktik (Glon)
# Kuadran I (0-90), II (90-180), III (180-270), IV (270-360)
def hitung_kuadran(df, kolom_glon):
    q1 = len(df[(df[kolom_glon] >= 0) & (df[kolom_glon] < 90)])
    q2 = len(df[(df[kolom_glon] >= 90) & (df[kolom_glon] < 180)])
    q3 = len(df[(df[kolom_glon] >= 180) & (df[kolom_glon] < 270)])
    q4 = len(df[(df[kolom_glon] >= 270) & (df[kolom_glon] < 360)])
    return q1, q2, q3, q4

# Menggunakan kolom bujur yang sesuai dari tiap file CSV
oc_kuadran = hitung_kuadran(oc_muda, '_Glon')
ob_kuadran = hitung_kuadran(ob_disk, 'GLON')

print("\n--- Data Pendukung No. 8: Jumlah Tracer per Kuadran ---")
print(f"Kuadran I   (0° - 90°)   [Arah Pusat]      : OC Muda = {oc_kuadran[0]:>4}, Bintang OB = {ob_kuadran[0]:>4}")
print(f"Kuadran II  (90° - 180°) [Arah Luar]       : OC Muda = {oc_kuadran[1]:>4}, Bintang OB = {ob_kuadran[1]:>4}")
print(f"Kuadran III (180° - 270°) [Arah Luar]      : OC Muda = {oc_kuadran[2]:>4}, Bintang OB = {ob_kuadran[2]:>4}")
print(f"Kuadran IV  (270° - 360°) [Arah Pusat]     : OC Muda = {oc_kuadran[3]:>4}, Bintang OB = {ob_kuadran[3]:>4}")
print("=============================================================================\n")


ANALISIS TAMBAHAN (NOMOR 7 & 8)
--- Data Pendukung No. 7: Karakteristik Tracer ---
Sebaran (Std Dev) sumbu Z OC Muda : 0.077 kpc
Sebaran (Std Dev) sumbu Z Btg OB  : 0.070 kpc
Semakin kecil nilainya, semakin objek tersebut menempel ketat pada piringan.

--- Data Pendukung No. 8: Jumlah Tracer per Kuadran ---
Kuadran I   (0° - 90°)   [Arah Pusat]      : OC Muda =   73, Bintang OB =  289
Kuadran II  (90° - 180°) [Arah Luar]       : OC Muda =   87, Bintang OB =  287
Kuadran III (180° - 270°) [Arah Luar]      : OC Muda =   91, Bintang OB =  174
Kuadran IV  (270° - 360°) [Arah Pusat]     : OC Muda =  113, Bintang OB =  632



In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Asumsi: oc_muda dan ob_disk sudah didefinisikan dari kodemu sebelumnya.
# Fungsi penghitung kuadran
def hitung_kuadran(df, kolom_glon):
    q1 = len(df[(df[kolom_glon] >= 0) & (df[kolom_glon] < 90)])
    q2 = len(df[(df[kolom_glon] >= 90) & (df[kolom_glon] < 180)])
    q3 = len(df[(df[kolom_glon] >= 180) & (df[kolom_glon] < 270)])
    q4 = len(df[(df[kolom_glon] >= 270) & (df[kolom_glon] < 360)])
    return [q1, q2, q3, q4]

oc_kuadran = hitung_kuadran(oc_muda, '_Glon')
ob_kuadran = hitung_kuadran(ob_disk, 'GLON')

# Pembuatan Bar Chart
labels = ['Kuadran I\n(0°-90°)\n[Arah Pusat]', 
          'Kuadran II\n(90°-180°)\n[Arah Luar]', 
          'Kuadran III\n(180°-270°)\n[Arah Luar]', 
          'Kuadran IV\n(270°-360°)\n[Arah Pusat]']

x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
rects1 = ax.bar(x - width/2, oc_kuadran, width, label='Gugus Terbuka (OC) Muda', color='#2ca02c', edgecolor='black')
rects2 = ax.bar(x + width/2, ob_kuadran, width, label='Bintang OB', color='#ff7f0e', edgecolor='black')

ax.set_ylabel('Jumlah Objek', fontsize=12)
ax.set_title('Perbandingan Jumlah Penjejak Spiral (Tracer) per Kuadran Galaktik', fontsize=14, pad=15)
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=11)
ax.legend(fontsize=12)
ax.grid(axis='y', linestyle='--', alpha=0.7)

# Fungsi untuk memunculkan angka di atas setiap batang
def autolabel(rects):
    for rect in rects:
        height = rect.get_height()
        ax.annotate(f'{height}',
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 3),  
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=11, fontweight='bold')

autolabel(rects1)
autolabel(rects2)

plt.tight_layout()
plt.savefig('grafik_kuadran_tracer.png', dpi=300)
print("Berhasil menyimpan grafik: grafik_kuadran_tracer.png")

Berhasil menyimpan grafik: grafik_kuadran_tracer.png


In [12]:
"""
=============================================================================
LANGKAH 7: Plot Posisi Bintang OB pada Bidang Galaksi
=============================================================================
Data yang dibutuhkan:
  - bintang_ob_final.csv   (Melnik & Efremov 1995)
  - ncovocc_final.csv      (NCOVOCC, Dias et al. 2003) ← untuk perbandingan

Output:
  - langkah7_OB_bidang_galaksi.png   (OB saja: tanpa & dengan model spiral)
  - langkah7_OC_vs_OB_spiral.png     (perbandingan OC muda vs OB)
=============================================================================
"""

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')       # Hapus baris ini jika ingin tampil interaktif
import matplotlib.pyplot as plt

plt.rcParams.update({'font.size': 11, 'axes.titlesize': 12, 'axes.labelsize': 11})

X0 = 8.0   # Jarak Matahari–Pusat Galaksi (kpc)

# =============================================================================
# 1. MUAT & BERSIHKAN DATA
# =============================================================================

# ── Bintang OB ────────────────────────────────────────────────────────────────
ob = pd.read_csv('bintang_ob_final.csv')
ob['Name'] = ob['Name'].astype(str).str.strip()
ob = ob.drop_duplicates(keep='first').reset_index(drop=True)
ob = ob[ob['rBH'] > 0].copy()                 # jarak harus positif

# ── Gugus Terbuka OC (hanya untuk panel perbandingan) ────────────────────────
oc = pd.read_csv('ncovocc_final.csv')
oc = oc.drop_duplicates(keep='first').reset_index(drop=True)
oc['has_age'] = oc['Age'].notna()

# =============================================================================
# 2. HITUNG KOORDINAT GALAKTOSENTRIS (X, Y, Z)
# =============================================================================
# Persamaan:
#   X = X₀ − d·cos(b)·cos(l)
#   Y = d·cos(b)·sin(l)
#   Z = d·sin(b)
#
# OB  → d = rBH (sudah dalam kpc)
# OC  → d = Dist / 1000 (parsec ke kpc)

# ── OB ───
l_ob = np.radians(ob['GLON'])
b_ob = np.radians(ob['GLAT'])
d_ob = ob['rBH']
ob['X'] = X0 - d_ob * np.cos(b_ob) * np.cos(l_ob)
ob['Y'] = d_ob * np.cos(b_ob) * np.sin(l_ob)
ob['Z'] = d_ob * np.sin(b_ob)

# ── OC ───
l_oc = np.radians(oc['_Glon'])
b_oc = np.radians(oc['_Glat'])
d_oc = oc['Dist'] / 1000.0
oc['X'] = X0 - d_oc * np.cos(b_oc) * np.cos(l_oc)
oc['Y'] = d_oc * np.cos(b_oc) * np.sin(l_oc)
oc['Z'] = d_oc * np.sin(b_oc)

# =============================================================================
# 3. SELEKSI |Z| < 0.3 kpc  +  batas plot soal
# =============================================================================

ob_disk = ob[ob['Z'].abs() < 0.3].copy()
oc_disk = oc[oc['Z'].abs() < 0.3].copy()
oc_muda = oc_disk[oc_disk['has_age'] & (oc_disk['Age'] < 7.5)]

def dalam_batas(df):
    """Filter 0 < X < 12 kpc dan |Y| < 6 kpc sesuai soal."""
    m = (df['X'] > 0) & (df['X'] < 12) & (df['Y'] > -6) & (df['Y'] < 6)
    return df[m]

ob_plot   = dalam_batas(ob_disk)
oc_m_plot = dalam_batas(oc_muda)

print(f"Bintang OB dalam batas plot : {len(ob_plot)}")
print(f"OC muda dalam batas plot    : {len(oc_m_plot)}")

# =============================================================================
# 4. MODEL LENGAN SPIRAL  r(θ) = r_i · exp(k · (θ − θ₀))
# =============================================================================

theta = np.linspace(-np.pi, 2.5 * np.pi, 3000)

LENGAN = [
    # (Nama, r_i kpc, k=tan(pitch), θ₀ rad, warna)
    ('Norma–Cygnus',  3.48, 0.26, 0.00, 'purple'),
    ('Scutum–Crux',   4.90, 0.26, 0.00, 'darkorange'),
    ('Carina–Sag.',   6.04, 0.23, 0.00, 'dodgerblue'),
    ('Local Arm',     7.70, 0.18, 4.80, 'forestgreen'),
    ('Perseus',       9.90, 0.21, 4.71, 'red'),
]

def kurva_spiral(r_i, k, theta0, theta_arr):
    r    = r_i * np.exp(k * (theta_arr - theta0))
    x_sp = r * np.cos(theta_arr)
    y_sp = r * np.sin(theta_arr)
    return x_sp, y_sp

def plot_spiral(ax):
    """Tambahkan semua lengan spiral ke axes."""
    for nama, r_i, k, t0, col in LENGAN:
        x_sp, y_sp = kurva_spiral(r_i, k, t0, theta)
        mask = (x_sp > 0) & (x_sp < 12) & (y_sp > -6) & (y_sp < 6)
        ax.plot(y_sp[mask], x_sp[mask],
                color=col, lw=2.0, alpha=0.85,
                label=nama, zorder=5)

def plot_referensi(ax):
    """Tambahkan tanda Matahari, GC, dan lingkaran jarak."""
    ax.plot(0, X0, '*', ms=18, color='gold', mec='darkorange',
            mew=1.5, zorder=8, label='☀ Matahari')
    ax.plot(0,  0, 'o', ms=10, color='black',
            zorder=8, label='● Pusat Galaksi')
    for r in [2, 4, 6, 8, 10]:
        ax.add_patch(plt.Circle((0, 0), r, color='gray',
                                fill=False, lw=0.6, ls='--', alpha=0.3))
        ax.text(r * 0.68, r * 0.68 + 0.15, f'{r} kpc',
                fontsize=7.5, color='gray', alpha=0.6)

def atur_axes(ax, judul):
    ax.set_xlabel('Y (kpc) → arah $l = 90°$')
    ax.set_ylabel('X (kpc) → arah $l = 0°$ (GC)')
    ax.set_title(judul)
    ax.set_xlim(-6, 6)
    ax.set_ylim(0, 12)
    ax.set_aspect('equal')
    ax.legend(loc='upper right', fontsize=8, framealpha=0.88)
    ax.grid(True, alpha=0.2)

# =============================================================================
# 5. PLOT A – Bintang OB saja (tanpa spiral | dengan spiral)
# =============================================================================

fig1, (axL, axR) = plt.subplots(1, 2, figsize=(17, 8))
fig1.suptitle(
    'Langkah 7: Distribusi Bintang OB pada Bidang Galaksi\n'
    'Kiri: tanpa model spiral  |  Kanan: dengan model lengan spiral',
    fontsize=13, fontweight='bold')

# Panel kiri – OB saja, warna = jarak heliosentris
sc = axL.scatter(
    ob_plot['Y'], ob_plot['X'],
    marker='x',
    s=20,
    c=ob_plot['rBH'],
    cmap='plasma_r',
    alpha=0.65,
    linewidths=0.9,
    zorder=3,
    label=f'Bintang OB (n={len(ob_plot)})'
)
plt.colorbar(sc, ax=axL, label='Jarak heliosentris $r_{BH}$ (kpc)', fraction=0.04)
plot_referensi(axL)
atur_axes(axL,
    '(a) Bintang OB – tanpa model spiral\n'
    '(warna = jarak dari Matahari)')

# Panel kanan – OB + model spiral
axR.scatter(
    ob_plot['Y'], ob_plot['X'],
    marker='x', s=20, c='crimson',
    alpha=0.6, linewidths=0.9,
    zorder=3, label=f'Bintang OB (n={len(ob_plot)})'
)
plot_spiral(axR)
plot_referensi(axR)
atur_axes(axR,
    '(b) Bintang OB + Model Lengan Spiral\n'
    '(Dias & Lépine 2005)')

fig1.tight_layout()
fig1.savefig('langkah7_OB_bidang_galaksi.png', dpi=150, bbox_inches='tight')
plt.close(fig1)
print("Disimpan: langkah7_OB_bidang_galaksi.png")

# =============================================================================
# 6. PLOT B – Perbandingan langsung OC Muda vs Bintang OB
# =============================================================================

fig2, (axA, axB) = plt.subplots(1, 2, figsize=(17, 8))
fig2.suptitle(
    'Langkah 7: Perbandingan Penanda Lengan Spiral\n'
    'Kiri: Gugus Terbuka Muda (OC)  |  Kanan: Bintang OB',
    fontsize=13, fontweight='bold')

# Panel kiri – OC muda
axA.scatter(
    oc_m_plot['Y'], oc_m_plot['X'],
    marker='o', s=20, c='steelblue',
    alpha=0.8, zorder=3,
    label=f'OC Muda log t<7.5 (n={len(oc_m_plot)})'
)
plot_spiral(axA)
plot_referensi(axA)
atur_axes(axA,
    f'(a) Gugus Terbuka Muda\n'
    f'log t < 7.5  (usia < 32 Myr, n={len(oc_m_plot)})')

# Panel kanan – Bintang OB
axB.scatter(
    ob_plot['Y'], ob_plot['X'],
    marker='x', s=20, c='crimson',
    alpha=0.7, linewidths=0.9,
    zorder=3, label=f'Bintang OB (n={len(ob_plot)})'
)
plot_spiral(axB)
plot_referensi(axB)
atur_axes(axB,
    f'(b) Bintang OB\n'
    f'(Melnik & Efremov 1995, n={len(ob_plot)})')

fig2.tight_layout()
fig2.savefig('langkah7_OC_vs_OB_spiral.png', dpi=150, bbox_inches='tight')
plt.close(fig2)
print("Disimpan: langkah7_OC_vs_OB_spiral.png")

Bintang OB dalam batas plot : 1382
OC muda dalam batas plot    : 356
Disimpan: langkah7_OB_bidang_galaksi.png
Disimpan: langkah7_OC_vs_OB_spiral.png
